# 03. GBDT 파라미터 가설 시험

02의 GBDT 설정(`max_iter=600, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0`)은 튜닝한 값이 아니라 흔히 쓰는 출발값이에요. 가설을 세우고 하나씩 시험해서 근거 있는 값으로 바꿔요.

| # | 가설 | 근거 |
|---|---|---|
| H1 | 나무 개수를 늘리면 오차가 줄어든다 | 02에서 일찍 멈추기가 한 번도 작동하지 않고 5개 fold 모두 상한 600개를 다 씀 → 덜 배운 상태 |
| H2 | 학습률을 낮추고 나무를 늘리면 조금 더 좋아진다 | 나무 하나가 고치는 폭이 작을수록 꼼꼼하게 맞춤 (대신 나무가 더 필요) |
| H3 | 나무를 크게(잎 수) 하면 좋아진다 | 하중은 강종 × 스탠드 × 두께 × 온도가 함께 작용 → 큰 나무가 여러 변수가 섞인 관계를 잘 잡음 |
| H4 | L2 규제는 정확도에 영향이 작다 | 약 4만 행이면 외워서 맞히기(과적합)가 작음 |
| H5 | 잎의 최소 데이터 수를 키우면 "더 누르면 하중이 줄어드는" 행이 줄어든다 | 잎마다 데이터가 많으면 예측이 덜 들쭉날쭉함 → 최적화에 유리 |
| H6 | 손실 함수를 `gamma`로 바꾸면 F5(작은 하중)가 좋아진다 | 물리식이 곱하기 구조 → 로그 공간에서 배우는 `gamma`와 잘 맞고, 오차를 비율로 다룸 |

### 시험 규칙
1. **한 번에 하나만 바꿔요.** 나머지는 기준 설정(BASE) 그대로 둬요.
2. **고르는 데이터와 평가하는 데이터를 나눠요.** 평가 점수를 보고 값을 고르면 그 점수가 부풀어요.

| 데이터 | 코일 | 쓰임 |
|---|---|---|
| 학습 | fold 0~3의 75% (전체의 60%) | 가설 시험 때 모델 학습 |
| 검증 | fold 0~3의 25% (전체의 20%) | 가설 시험 때 점수 비교 → **값 고르기** |
| 최종 평가 | fold 4 (전체의 20%) | 마지막에 **한 번만**: 02 설정 vs 고른 설정 |

fold는 01·02와 같은 코일 묶음이에요. **전체 실행에 9분쯤 걸려요**(대부분 H1·H2의 나무 3,000개 곡선).

## 0. 준비

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

installed = {f.name for f in font_manager.fontManager.ttflist}
for font in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if font in installed:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlesize": 12,
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})

# 데이터: kdh 전처리 함수 (01·02와 같음)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kdh/src/prepare_data.py").exists())
sys.path.insert(0, str(REPO / "kdh/src"))
sys.dont_write_bytecode = True   # kdh 폴더에 캐시(__pycache__)를 만들지 않음
from prepare_data import COLS, add_features, add_chain_flag

raw = pd.read_csv(REPO / "main/raw_data/metals.csv", usecols=list(COLS)).rename(columns=COLS)
df = add_chain_flag(add_features(raw))

V1, HO1 = 1096.0, 27.7

def add_physics(d):
    d = d.copy()
    d["dh"] = d["h_in"] - d["h_out"]
    d["reduction"] = d["dh"] / d["h_in"]
    d["L"] = np.sqrt(d["R"] * d["dh"])
    d["strain"] = np.log(d["h_in"] / d["h_out"])
    d["v"] = V1 * HO1 / d["h_out"]
    d["strain_rate"] = d["strain"] * d["v"] / d["L"]
    return d

df = add_physics(df)

# 단조성 점검용: 압하량 5% 더 누른 가상 데이터 (02와 같음)
df_more = df.copy()
df_more["h_out"] = df["h_out"] - 0.05 * df["dh"]
df_more = add_physics(df_more)

# 01·02와 같은 코일 단위 5-fold
df["fold"] = -1
for k, (_, test_idx) in enumerate(GroupKFold(n_splits=5).split(df, groups=df["coil"])):
    df.loc[df.index[test_idx], "fold"] = k

NUM = ["stand_no", "h_in", "h_out", "W", "R", "T_K", "reduction", "dh", "L", "strain", "strain_rate"]
FEATURES = ["grade"] + NUM
GRADES = sorted(df["grade"].unique())

def tree_input(d):
    X = d[FEATURES].copy()
    X["grade"] = pd.Categorical(X["grade"], categories=GRADES)
    return X

### 데이터 나누기

In [ ]:
dev = df[df["fold"] != 4]     # 값 고르기용 (80%)
test = df[df["fold"] == 4]    # 최종 평가용 (20%), 마지막까지 손대지 않음

tr_idx, va_idx = next(GroupKFold(n_splits=4).split(dev, groups=dev["coil"]))
train, valid = dev.iloc[tr_idx], dev.iloc[va_idx]

split_view = pd.DataFrame({
    "코일 수": [train["coil"].nunique(), valid["coil"].nunique(), test["coil"].nunique()],
    "행 수": [len(train), len(valid), len(test)],
}, index=["학습", "검증", "최종 평가"])
display(split_view)
assert not set(train["coil"]) & set(valid["coil"]) and not set(dev["coil"]) & set(test["coil"])

### 기준 설정(BASE)과 시험 함수

- BASE는 02의 값과 같아요. 다만 **일찍 멈추기는 꺼요.** 02에서도 한 번도 작동하지 않았고, 켜 두면 학습 데이터 10%를 행 단위로 떼어 내서(같은 코일이 섞임) 비교가 흐려지기 때문이에요.
- `run()`은 설정 하나를 학습 데이터로 학습하고, 검증 데이터로 점수를 매겨 기록표(`log`)에 한 줄 추가해요.

In [ ]:
BASE = dict(max_iter=600, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
            l2_regularization=1.0, early_stopping=False)
log = []

def make_model(**changes):
    return HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0, **{**BASE, **changes})

def run(hypothesis, label, **changes):
    start = time.time()
    model = make_model(**changes).fit(tree_input(train), train["F_MN"])
    fit_sec = time.time() - start
    p_valid = model.predict(tree_input(valid))
    p_more = model.predict(tree_input(df_more.loc[valid.index]))
    row = {
        "가설": hypothesis, "설정": label,
        "학습 MAE": mean_absolute_error(train["F_MN"], model.predict(tree_input(train))),
        "검증 MAE": mean_absolute_error(valid["F_MN"], p_valid),
        "검증 R²": r2_score(valid["F_MN"], p_valid),
        "더 누르면 감소 (%)": (p_more < p_valid - 1e-9).mean() * 100,
        "학습 시간 (초)": fit_sec,
    }
    log.append(row)
    print(f"[{hypothesis}] {label:28s} 검증 MAE {row['검증 MAE']:.3f}  학습 MAE {row['학습 MAE']:.3f}  "
          f"감소 {row['더 누르면 감소 (%)']:.1f}%  {fit_sec:.0f}초")
    return model, p_valid

FMT = {"학습 MAE": "{:.3f}", "검증 MAE": "{:.3f}", "검증 R²": "{:.4f}", "더 누르면 감소 (%)": "{:.1f}", "학습 시간 (초)": "{:.0f}"}
base_model, base_valid = run("BASE", "02 설정 (일찍 멈추기 끔)")

## H1·H2. 나무 개수와 학습률

학습률 3가지(0.1, 0.05, 0.02)로 나무를 3,000개까지 쌓으면서, **나무가 하나 늘 때마다** 검증 MAE를 계산해요(`staged_predict`). 곡선이 언제 평평해지는지 보면 필요한 나무 개수를 알 수 있어요.

In [ ]:
curves, fit_secs = {}, {}
for lr in [0.1, 0.05, 0.02]:
    start = time.time()
    model = make_model(learning_rate=lr, max_iter=3000).fit(tree_input(train), train["F_MN"])
    fit_secs[lr] = time.time() - start
    curves[lr] = np.array([mean_absolute_error(valid["F_MN"], p) for p in model.staged_predict(tree_input(valid))])
    print(f"학습률 {lr}: {fit_secs[lr]:.0f}초")

rows = []
for lr, c in curves.items():
    best = int(c.argmin())
    near = int(np.argmax(c <= c.min() * 1.01))   # 최저 MAE의 1% 안에 처음 들어온 나무 수
    rows.append({"학습률": lr, "나무 600개 MAE": c[599], "최저 MAE": c[best], "최저일 때 나무 수": best + 1,
                 "최저+1% 안에 든 나무 수": near + 1, "3000개 학습 시간 (초)": fit_secs[lr]})
curve_table = pd.DataFrame(rows).set_index("학습률")
display(curve_table.style.format({"나무 600개 MAE": "{:.3f}", "최저 MAE": "{:.3f}", "3000개 학습 시간 (초)": "{:.0f}"}))

fig, ax = plt.subplots(figsize=(8, 4.2))
for (lr, c), color in zip(curves.items(), [BLUE, ORANGE, AQUA]):
    ax.plot(np.arange(1, len(c) + 1), c, color=color, linewidth=2, label=f"학습률 {lr}")
ax.axvline(600, color="#898781", linestyle="--", linewidth=1)
low = min(c.min() for c in curves.values())
ax.set_ylim(low * 0.95, low * 1.6)
ax.text(620, low * 1.55, "02 설정 (600개)", color="#52514e", va="top")
ax.set_xlabel("나무 개수")
ax.set_ylabel("검증 MAE (MN)")
ax.set_title("나무가 늘어날 때 검증 오차")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

- **H1 맞음.** 02 설정(학습률 0.05)은 나무 600개에서 검증 MAE 0.470인데, 3,000개까지 계속 내려가 0.421이 돼요. 600개는 덜 배운 상태였어요.
- **H2 틀림.** 학습률을 낮출수록 같은 나무 수에서 오차가 커요. 3,000개 안에서는 학습률 0.1이 가장 좋고(최저 0.417), 0.02는 3,000개로도 0.438까지만 내려가요. 학습률을 낮춰서 좋아지려면 나무가 훨씬 더 필요해서 시간 대비 손해예요.
- 학습률 0.1은 나무 1,520개에서 최저 MAE의 1% 안에 들어오고, 그 뒤로는 거의 평평해요. → **학습률 0.1, 나무 1,500개**를 골라요.

## H3. 나무 크기 (`max_leaf_nodes`)

나무 개수(600)와 학습률(0.05)은 BASE 그대로 두고 잎 수만 바꿔요. **학습 MAE와 검증 MAE의 차이**가 크게 벌어지면 외워서 맞히기(과적합)가 시작된 거예요.

In [ ]:
for leaves in [15, 63, 127]:
    run("H3 나무 크기", f"max_leaf_nodes={leaves}", max_leaf_nodes=leaves)

- **정확도만 보면 H3 맞음.** 잎 63개 0.437, 127개 0.422로 BASE(0.470)보다 좋아요. 잎 15개는 0.519로 나빠요.
- 하지만 대가가 커요.
  - **과적합 신호:** 학습 MAE와 검증 MAE의 차이가 0.06(잎 31) → 0.11(63) → 0.17(127)로 벌어져요.
  - **단조성 악화:** 더 누르면 하중이 줄어드는 행이 5.2% → 6.3% → 7.6%로 늘어요.
  - **시간:** 잎 127개는 학습이 4배(17초 → 67초)예요.
- → 최적화에 쓸 모델이라 **잎 31개를 유지**하고, 정확도는 나무 개수로 올려요(H1).

## H4. L2 규제 (`l2_regularization`)

In [ ]:
for l2 in [0.0, 10.0]:
    run("H4 L2 규제", f"l2_regularization={l2}", l2_regularization=l2)

- **H4 맞음.** L2 규제를 0, 1, 10으로 바꿔도 검증 MAE가 0.470~0.472로 같아요. 1.0을 그대로 둬요.

## H5. 잎의 최소 데이터 수 (`min_samples_leaf`)와 단조성

정확도(검증 MAE)와 함께 **더 누르면 하중이 줄어드는 행 비율**을 봐요.

In [ ]:
for leaf_min in [20, 200, 500]:
    run("H5 잎 최소 데이터", f"min_samples_leaf={leaf_min}", min_samples_leaf=leaf_min)

- **H5 틀림.** 잎 최소 데이터를 200, 500으로 키우면 정확도만 나빠지고(0.487, 0.514), 더 누르면 하중이 줄어드는 행은 5.0~5.5%로 그대로예요.
- → 규제로는 단조성을 고칠 수 없어요. **다음 단계에서 단조 제약(`monotonic_cst`)을 직접 넣어야 해요.**
- 20으로 줄여도 0.467로 BASE(0.470)와 차이가 작아서 40을 유지해요.

## H6. 손실 함수 `gamma`

`gamma`는 하중의 로그를 배우는 것과 비슷해요. 그래서 큰 하중(F1)과 작은 하중(F5)의 오차를 비율로 비슷하게 다뤄요. 스탠드별 오차율로 비교해요.

In [ ]:
gamma_model, gamma_valid = run("H6 손실 함수", "loss=gamma", loss="gamma")

err = pd.DataFrame({
    "BASE (squared_error)": (base_valid - valid["F_MN"]).abs() / valid["F_MN"] * 100,
    "gamma": (gamma_valid - valid["F_MN"]).abs() / valid["F_MN"] * 100,
}, index=valid.index)
print("검증 데이터 스탠드별 평균 오차율 (%)")
display(err.groupby(valid["stand"]).mean().round(2))

- **H6 절반만 맞음.** F3~F5 오차율은 조금 줄었지만(F5 3.19% → 3.04%), F1·F2는 늘어서 전체 검증 MAE는 0.474로 BASE(0.470)보다 조금 나빠요.
- 하중이 큰 F1·F2가 최적화의 핵심이라 `squared_error`를 유지해요.

## 가설 시험 기록 (검증 데이터)

In [ ]:
log_table = pd.DataFrame(log)
display(log_table.style.format(FMT))

## 최종 평가: 02 설정 vs 고른 설정

고른 설정은 **학습률 0.1, 나무 1,500개**이고 나머지는 02와 같아요. 나무 수를 곡선으로 정했으므로 일찍 멈추기는 꺼요.

| 값 | 02 설정 | 고른 값 | 근거 |
|---|---|---|---|
| `learning_rate` | 0.05 | **0.1** | H2: 같은 나무 수에서 오차가 가장 낮음 |
| `max_iter` | 600 | **1,500** | H1: 최저 오차의 1% 안에 드는 나무 수 |
| `max_leaf_nodes` | 31 | 31 | H3: 키우면 정확도는 오르지만 과적합·단조성 악화, 시간 증가 |
| `min_samples_leaf`, `l2_regularization` | 40, 1.0 | 40, 1.0 | H4·H5: 영향이 없거나 나빠짐 |
| `loss` | squared_error | squared_error | H6: 전체 오차가 조금 나빠짐 |

두 설정을 **값 고르기용 데이터 전체(fold 0~3)** 로 학습하고, 지금까지 손대지 않은 **fold 4**로 한 번만 평가해요. 최적화에서 모델을 수천 번 부르므로 1만 행 예측 시간도 재요.

In [ ]:
# 가설 시험 결과로 고른 설정 (위 "정리" 근거)
CHOSEN_PARAMS = dict(max_iter=1500, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=40,
                     l2_regularization=1.0, early_stopping=False)

FINAL = {
    "02 설정": dict(max_iter=600, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
                    l2_regularization=1.0, early_stopping=True, n_iter_no_change=30),
    "고른 설정": CHOSEN_PARAMS,
}
sample = tree_input(test.sample(10_000, replace=True, random_state=0))
rows = []
for name, params in FINAL.items():
    start = time.time()
    model = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0, **params).fit(tree_input(dev), dev["F_MN"])
    fit_sec = time.time() - start
    p = model.predict(tree_input(test))
    p_more = model.predict(tree_input(df_more.loc[test.index]))
    start = time.time()
    model.predict(sample)
    rows.append({"설정": name, "최종 MAE": mean_absolute_error(test["F_MN"], p), "최종 R²": r2_score(test["F_MN"], p),
                 "평균 오차율 (%)": ((p - test["F_MN"]).abs() / test["F_MN"]).mean() * 100,
                 "더 누르면 감소 (%)": (p_more < p - 1e-9).mean() * 100,
                 "학습 시간 (초)": fit_sec, "1만 행 예측 (초)": time.time() - start})
final_table = pd.DataFrame(rows).set_index("설정")
display(final_table.style.format({"최종 MAE": "{:.3f}", "최종 R²": "{:.4f}", "평균 오차율 (%)": "{:.2f}",
                                  "더 누르면 감소 (%)": "{:.1f}", "학습 시간 (초)": "{:.0f}", "1만 행 예측 (초)": "{:.2f}"}))

## 정리

**최종 평가 (fold 4, 처음 보는 코일 1,559개)**

| | 02 설정 | 고른 설정 |
|---|---|---|
| MAE | 0.460 MN | **0.389 MN (−15%)** |
| R² | 0.9916 | **0.9935** |
| 평균 오차율 | 2.59% | **2.18%** |
| 더 누르면 하중 감소 | 5.2% | 6.9% ⚠️ |
| 학습 시간 | 19초 | 41초 |
| 1만 행 예측 | 0.16초 | 0.35초 |

- **가설 결과:** H1(나무 개수) 맞음, H2(학습률 낮추기) 틀림, H3(나무 크기) 정확도는 맞지만 대가가 큼, H4(L2) 맞음(영향 없음), H5(규제로 단조성 개선) 틀림, H6(gamma) 절반만 맞음.
- **정확도:** 처음 보는 코일에서 오차가 15% 더 줄었어요. 예측도 1만 행에 0.35초라 최적화·시뮬레이터에 쓰기에 충분히 빨라요.
- **남은 문제:** 나무가 많아지면 더 잘게 맞추면서, 더 누르면 하중이 줄어드는 행도 늘어요(5.2% → 6.9%). H5에서 본 것처럼 규제로는 안 고쳐지니, **다음 단계에서 단조 제약을 넣어** 정확도와 단조성을 함께 확인해요.
- **02 반영 여부:** 02의 GBDT와 하이브리드(GBDT로 보정)는 아직 예전 설정이에요. 고른 설정으로 바꾸면 둘 다 좋아질 것으로 보여요.